# Lab 5 — List of Improving Moves in Steepest Local Search for Selective TSP with Node Costs

Authors: Wiktor Talarek 160225, Daniel Skalski 160261

This notebook implements the Lab 5 assignment: improve the time efficiency of the **steepest local search**
by reusing move evaluations (deltas) from previous iterations through a **list of improving moves (LM)**.
The neighbourhood is the one that proved best in Lab 3 / used in Lab 4: **2-opt intra-route moves + inter-route node exchange**.
The baseline is the full steepest local search without the LM mechanism.

## 1. Description of the problem

We are given a set of `n = 200` nodes (instances `TSPA`, `TSPB` in `data/TSPA.csv`, `data/TSPB.csv`).
Each row is `x;y;cost` (the files have no header line):

- `(x, y)` — position in the plane,
- `cost[i] >= 0` — price paid if node `i` is visited.

Goal: **select exactly `k = ceil(n / 2) = 100` nodes and find a Hamiltonian cycle** through them minimizing:

```
f(tour) = sum_i dist[tour[i], tour[(i+1) mod k]] + sum_i cost[tour[i]]
```

where `dist` is the Euclidean distance **rounded to the nearest integer**:

```
dist[i,j] = round(sqrt((x_i-x_j)^2 + (y_i-y_j)^2))
```

Methodological constraints (same as in Labs 1-4):

- The distance matrix is precomputed once after loading the instance.
- Optimization methods access only `dist` and `costs`, never `(x, y)`. Coordinates are used only for
  visualization.
- Every move delta is computed in `O(1)`; no neighbour solution is built explicitly and the objective
  is never recomputed from scratch inside the search.

This is a **Selective / Orienteering-like TSP**: subset selection and permutation are optimized jointly.

## 2. Helpers and parameters

In [1]:
%matplotlib inline
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable

DATA = Path('data')
FIG = Path('figures')
SOL = Path('solutions')
INSTANCES = ['TSPA', 'TSPB']
RNG_SEED = 0
NUM_RUNS = 200


def load_instance(name: str):
    d = np.loadtxt(DATA / f'{name}.csv', delimiter=';', dtype=int)
    xy, costs = d[:, :2], d[:, 2]
    diff = xy[:, None, :] - xy[None, :, :]
    dist = np.rint(np.sqrt((diff ** 2).sum(axis=-1))).astype(np.int64)
    n = len(costs)
    k = (n + 1) // 2
    return {'name': name, 'xy': xy, 'costs': costs, 'dist': dist, 'n': n, 'k': k}


def compute_cost(tour, dist, costs) -> int:
    t = np.asarray(tour, dtype=int)
    return int(dist[t, np.roll(t, -1)].sum() + costs[t].sum())


def check_solution(tour, n: int, k: int):
    t = list(tour)
    if len(t) != k:
        return False, f'length {len(t)} != k={k}'
    if len(set(t)) != k:
        return False, 'duplicate nodes'
    if any(v < 0 or v >= n for v in t):
        return False, 'index out of range'
    return True, 'ok'


def random_tour(n: int, k: int, rng: np.random.Generator) -> list[int]:
    selected = rng.choice(n, size=k, replace=False)
    rng.shuffle(selected)
    return [int(v) for v in selected]


problems = {name: load_instance(name) for name in INSTANCES}
print('loaded instances:', {n: (problems[n]['n'], problems[n]['k']) for n in INSTANCES})


loaded instances: {'TSPA': (200, 100), 'TSPB': (200, 100)}


## 3. Implemented algorithms — pseudocode

Common notation: `n` nodes, select `k`, `D` distance matrix, `C` node costs, `used[]` visited flag,
`tour[]` the current cyclic tour.

The neighbourhood is **2-opt intra-route moves + inter-route node exchange** (the best Lab 3 neighbourhood,
reused in Lab 4).

### 3.1 Baseline: full steepest local search

```text
STEEPEST_FULL(tour, used, D, C, k, n):
    repeat:
        best_delta <- 0; best_move <- none
        # inter-route: exchange one selected node with one unselected node
        for each selected position p and each unselected node u:
            d <- D[prev(p),u] + D[u,next(p)] - D[prev(p),tour[p]] - D[tour[p],next(p)] + C[u] - C[tour[p]]
            if d < best_delta: best_delta <- d; best_move <- (inter, p, u)
        # intra-route: 2-opt
        for each non-adjacent pair (i,j):
            d <- D[tour[i],tour[j]] + D[tour[i+1],tour[j+1]] - D[tour[i],tour[i+1]] - D[tour[j],tour[j+1]]
            if d < best_delta: best_delta <- d; best_move <- (2opt, i, j)
        if best_move is none: break
        apply best_move; cost <- cost + best_delta
    return tour, cost
```

The whole neighbourhood (`k(n-k) = 10 000` inter-route + `4 850` 2-opt deltas per iteration) is scanned
at every iteration.

### 3.2 Steepest local search with a list of improving moves (LM)

LM keeps improving moves sorted from the best (most negative delta) to the worst. Each move stores the
two edges it removes (with their saved direction) and enough information to apply it.

When a saved move is browsed, three situations are distinguished (the removed edges are the ones defining
the saved move):

1. **Removed edges no longer exist in the current solution (at least one of them)** — remove the move from LM.
2. **Removed edges exist in a different relative direction** than saved — leave the move in LM, do not apply
   it, keep browsing.
3. **Removed edges exist in the same relative direction (both reversed is also fine)** — apply the move and
   remove it from LM.

For an inter-route node exchange the move also requires the incoming node `u` to be currently unselected;
if `u` was selected by a later move the saved move is stale and is removed from LM (case 1).

```text
LM_STEEPEST(tour, used, D, C, k, n):
    LM <- empty list of improving moves, sorted by delta ascending
    Generate an initial solution x
    Evaluate all moves applicable to x; add every improving move (delta < 0) to LM
    repeat:
        for moves m from LM starting from the best until an applicable move is found:
            Check if m is applicable (the 3 cases above); if not, remove it from LM
            if an applicable move m has been found:
                x <- m(x); cost <- cost + delta(m); remove m from LM
                (restart scanning LM from the best)
        if no applicable move was found:
            Evaluate all moves applicable to x; replace LM with the improving moves found
    until LM is empty
    return x, cost
```

New moves are evaluated with their edges in the direction they appear in the current tour, so moves whose
edges were reversed by a previous 2-opt are matched by case 3 ("both reversed"). The LM mechanism is used
separately from candidate moves (candidate moves are not part of this assignment).

The only randomness is the random starting solution (`k` distinct nodes sampled uniformly without
replacement and shuffled).

### 3.3 Baseline implementation

In [2]:
def steepest_ls_opt(tour, used, D, C, k, n):
    """Full steepest local search: inter-route node exchange + 2-opt intra-route moves."""
    t = np.array(tour, dtype=int).copy()
    used = used.copy()
    cost = compute_cost(t, D, C)
    iters = 0
    while True:
        iters += 1
        pos = np.arange(k)
        prev = t[(pos - 1) % k]
        nxt = t[(pos + 1) % k]
        s = t[pos]
        U = np.flatnonzero(~used)
        m = len(U)
        best_delta = 0
        best_move = None
        if m > 0:
            inter = (D[prev[:, None], U[None, :]]
                     + D[U[None, :], nxt[:, None]]
                     - (D[prev, s] + D[s, nxt])[:, None]
                     + C[U][None, :] - C[s][:, None])
            f = int(np.argmin(inter))
            d = int(inter.flat[f])
            if d < best_delta:
                best_delta = d
                best_move = ('inter', f // m, int(U[f % m]))
        ii2, jj2 = np.triu_indices(k, 2)
        v2 = ~((ii2 == 0) & (jj2 == k - 1))
        if v2.any():
            i3 = ii2[v2]; j3 = jj2[v2]
            aa = t[i3]; bb = t[(i3 + 1) % k]
            cc = t[j3]; dd2 = t[(j3 + 1) % k]
            twoopt = D[aa, cc] + D[bb, dd2] - D[aa, bb] - D[cc, dd2]
            f = int(np.argmin(twoopt)); d = int(twoopt[f])
            if d < best_delta:
                best_delta = d
                best_move = ('2opt', int(i3[f]), int(j3[f]))
        if best_move is None:
            break
        if best_move[0] == 'inter':
            _, p, u = best_move
            used[t[p]] = False
            used[u] = True
            t[p] = u
        else:
            _, i, j = best_move
            t[i + 1:j + 1] = t[i + 1:j + 1][::-1]
        cost += best_delta
    return t.tolist(), int(cost), iters


### 3.4 List-of-improving-moves implementation

In [3]:
def reverse_segment(t, start, end, k):
    """Reverse t[start..end] inclusive, walking forward modulo k."""
    idxs = []
    cur = start
    while True:
        idxs.append(cur)
        if cur == end:
            break
        cur = (cur + 1) % k
    vals = [t[idx] for idx in idxs][::-1]
    for idx, v in zip(idxs, vals):
        t[idx] = v


def evaluate_all_moves(t, used, D, C, k, n):
    """Evaluate the whole neighbourhood. Return improving moves (delta < 0) sorted by delta ascending."""
    moves = []
    t = np.asarray(t, dtype=int)
    pos = np.arange(k)
    prev = t[(pos - 1) % k]
    nxt = t[(pos + 1) % k]
    s = t[pos]
    U = np.flatnonzero(~used)
    m = len(U)
    if m > 0:
        inter = (D[prev[:, None], U[None, :]]
                 + D[U[None, :], nxt[:, None]]
                 - (D[prev, s] + D[s, nxt])[:, None]
                 + C[U][None, :] - C[s][:, None])
        flat = inter.ravel()
        neg = np.flatnonzero(flat < 0)
        P = np.repeat(pos, m)
        S = np.repeat(s, m)
        UU = np.tile(U, k)
        PP = np.repeat(prev, m)
        PN = np.repeat(nxt, m)
        for q in neg:
            p = int(P[q]); node_s = int(S[q]); u = int(UU[q])
            pv = int(PP[q]); nx = int(PN[q]); d = int(flat[q])
            moves.append({'kind': 'inter', 'p': p, 's': node_s, 'u': u,
                          'prev': pv, 'nxt': nx, 'delta': d,
                          'e1': (pv, node_s), 'e2': (node_s, nx)})
    ii2, jj2 = np.triu_indices(k, 2)
    valid = ~((ii2 == 0) & (jj2 == k - 1))
    ii2 = ii2[valid]; jj2 = jj2[valid]
    if len(ii2):
        a = t[ii2]; b = t[(ii2 + 1) % k]
        c = t[jj2]; dd = t[(jj2 + 1) % k]
        twoopt = D[a, c] + D[b, dd] - D[a, b] - D[c, dd]
        neg = np.flatnonzero(twoopt < 0)
        for q in neg:
            moves.append({'kind': '2opt', 'delta': int(twoopt[q]),
                          'e1': (int(a[q]), int(b[q])), 'e2': (int(c[q]), int(dd[q]))})
    moves.sort(key=lambda mv: mv['delta'])
    return moves


def move_status(mv, t, pos, k, used):
    """Return 1 (not applicable, remove from LM), 2 (different direction, keep), 3 (applicable)."""
    if mv['kind'] == 'inter':
        p = mv['p']; s = mv['s']; pv = mv['prev']; nx = mv['nxt']; u = mv['u']
        if t[p] == s and t[(p - 1) % k] == pv and t[(p + 1) % k] == nx and used[s] and not used[u]:
            return 3
        return 1
    a, b = mv['e1']; c, d = mv['e2']
    if a not in pos or b not in pos or c not in pos or d not in pos:
        return 1
    pa, pb, pc, pd = pos[a], pos[b], pos[c], pos[d]
    e1_fwd = (pb == (pa + 1) % k)
    e1_bwd = (pa == (pb + 1) % k)
    e2_fwd = (pd == (pc + 1) % k)
    e2_bwd = (pc == (pd + 1) % k)
    if e1_fwd and e2_fwd:
        if pa == pc or (pa + 1) % k == pc or pa == (pc + 1) % k:
            return 1
        return 3
    if e1_bwd and e2_bwd:
        if pb == pd or (pb + 1) % k == pd or pb == (pd + 1) % k:
            return 1
        return 3
    e1_exists = e1_fwd or e1_bwd
    e2_exists = e2_fwd or e2_bwd
    if not e1_exists or not e2_exists:
        return 1
    return 2


def apply_move(mv, t, used, k, pos):
    if mv['kind'] == 'inter':
        p = mv['p']; s = mv['s']; u = mv['u']
        used[s] = False
        used[u] = True
        t[p] = u
    else:
        a, b = mv['e1']; c, d = mv['e2']
        pa, pb = pos[a], pos[b]
        pc, pd = pos[c], pos[d]
        e1_fwd = (pb == (pa + 1) % k)
        e1_bwd = (pa == (pb + 1) % k)
        e2_fwd = (pd == (pc + 1) % k)
        e2_bwd = (pc == (pd + 1) % k)
        if e1_fwd and e2_fwd:
            start = (pa + 1) % k
            end = pc
        else:
            start = (pb + 1) % k
            end = pd
        reverse_segment(t, start, end, k)


def lm_steepest(init, used, D, C, k, n):
    """Steepest local search reusing deltas via a list of improving moves (LM)."""
    t = [int(v) for v in init]
    used = used.copy()
    cost = int(compute_cost(np.array(t), D, C))
    LM = evaluate_all_moves(t, used, D, C, k, n)
    n_inter = k * max(n - k, 0)
    n_2opt = k * (k - 1) // 2 - k - 1
    full_pass = n_inter + n_2opt
    n_evals = full_pass
    n_applies = 0
    while True:
        pos = {int(v): int(i) for i, v in enumerate(t)}
        applied = False
        i = 0
        while i < len(LM):
            mv = LM[i]
            st = move_status(mv, t, pos, k, used)
            if st == 1:
                LM.pop(i)
            elif st == 2:
                i += 1
            else:
                apply_move(mv, t, used, k, pos)
                cost += mv['delta']
                LM.pop(i)
                n_applies += 1
                applied = True
                break
        if applied:
            continue
        new_LM = evaluate_all_moves(t, used, D, C, k, n)
        n_evals += full_pass
        if not new_LM:
            break
        LM = new_LM
    return t, int(cost), n_applies, n_evals


In [ ]:
def steepest_ls_opt_fair(tour, used, D, C, k, n):
    t = list(tour)
    used = used.copy()
    cost = int(compute_cost(t, D, C))
    iters = 0

    while True:
        iters += 1
        best_delta = 0
        best_move = None

        for p in range(k):
            s = t[p]
            pv = t[(p - 1) % k]
            nx = t[(p + 1) % k]
            for u in range(n):
                if not used[u]:
                    d = D[pv, u] + D[u, nx] - D[pv, s] - D[s, nx] + C[u] - C[s]
                    if d < best_delta:
                        best_delta = d
                        best_move = ('inter', p, s, u)

        for i in range(k):
            for j in range(i + 2, k):
                if i == 0 and j == k - 1:
                    continue
                a, b = t[i], t[(i + 1) % k]
                c, dd = t[j], t[(j + 1) % k]
                d = D[a, c] + D[b, dd] - D[a, b] - D[c, dd]
                if d < best_delta:
                    best_delta = d
                    best_move = ('2opt', i, j, a, b, c, dd)

        if best_move is None:
            break

        if best_move[0] == 'inter':
            _, p, s, u = best_move
            used[s] = False
            used[u] = True
            t[p] = u
        else:
            _, i, j, _, _, _, _ = best_move
            t[i + 1:j + 1] = t[i + 1:j + 1][::-1]

        cost += best_delta
    return t, cost, iters

def evaluate_all_moves_fair(t, used, D, C, k, n):
    moves = []
    for p in range(k):
        s = t[p]
        pv = t[(p - 1) % k]
        nx = t[(p + 1) % k]
        for u in range(n):
            if not used[u]:
                d = D[pv, u] + D[u, nx] - D[pv, s] - D[s, nx] + C[u] - C[s]
                if d < 0:
                    moves.append({'kind': 'inter', 'p': p, 's': s, 'u': u,
                                  'prev': pv, 'nxt': nx, 'delta': d,
                                  'e1': (pv, s), 'e2': (s, nx)})
    for i in range(k):
        for j in range(i + 2, k):
            if i == 0 and j == k - 1:
                continue
            a, b = t[i], t[(i + 1) % k]
            c, dd = t[j], t[(j + 1) % k]
            d = D[a, c] + D[b, dd] - D[a, b] - D[c, dd]
            if d < 0:
                moves.append({'kind': '2opt', 'delta': d,
                              'e1': (a, b), 'e2': (c, dd)})
    moves.sort(key=lambda mv: mv['delta'])
    return moves

def lm_steepest_fair(init, used, D, C, k, n):
    t = list(init)
    used = used.copy()
    cost = int(compute_cost(t, D, C))
    LM = evaluate_all_moves_fair(t, used, D, C, k, n)
    n_applies = 0
    n_evals = (k * (n - k)) + (k * (k - 1) // 2 - k)

    while True:
        pos = {v: i for i, v in enumerate(t)}
        applied = False
        i = 0
        while i < len(LM):
            mv = LM[i]
            st = move_status(mv, t, pos, k, used)
            if st == 1:
                LM.pop(i)
            elif st == 2:
                i += 1
            else:
                apply_move(mv, t, used, k, pos)
                cost += mv['delta']
                LM.pop(i)
                n_applies += 1
                applied = True
                break
        if applied:
            continue
        new_LM = evaluate_all_moves_fair(t, used, D, C, k, n)
        n_evals += (k * (n - k)) + (k * (k - 1) // 2 - k)
        if not new_LM:
            break
        LM = new_LM
    return t, cost, n_applies, n_evals


## 4. Computational experiment

For each instance, 200 random starting solutions are generated with a fixed seed and reused by every
method (paired comparison). Each method is run 200 times. Times are total wall-clock seconds for the 200
runs and include random-start construction. The baseline is the full steepest local search without the LM
mechanism.

In [4]:
results = {}

for name in INSTANCES:
    pr = problems[name]
    n, k, D, C = pr['n'], pr['k'], pr['dist'], pr['costs']
    rng = np.random.default_rng(RNG_SEED)
    starts = [random_tour(n, k, rng) for _ in range(NUM_RUNS)]

    n_inter = k * (n - k)
    n_2opt = k * (k - 1) // 2 - k - 1
    full_pass_evals = n_inter + n_2opt

    results[name] = {'n': n, 'k': k, 'methods': {}}

    # --- baseline: full steepest local search ---
    t0 = time.perf_counter()
    b_costs, b_tours, b_iters = [], [], []
    for init in starts:
        used = np.zeros(n, bool)
        used[np.array(init)] = True
        tour, cost, it = steepest_ls_opt(init, used.copy(), D, C, k, n)
        b_costs.append(cost)
        b_tours.append(tour)
        b_iters.append(it)
    b_costs = np.array(b_costs)
    bj = int(np.argmin(b_costs))
    results[name]['methods']['baseline_full'] = {
        'costs': b_costs, 'mean': float(b_costs.mean()),
        'min': int(b_costs.min()), 'max': int(b_costs.max()),
        'best_tour': b_tours[bj], 'best_cost': int(b_costs.min()),
        'iters_mean': float(np.mean(b_iters)),
        'evals_mean': float(np.mean(b_iters)) * full_pass_evals,
        'time': time.perf_counter() - t0,
    }
    print(f"{name} baseline: mean={b_costs.mean():.1f} ({b_costs.min()}-{b_costs.max()}) "
          f"total={results[name]['methods']['baseline_full']['time']:.2f}s "
          f"iters/run={np.mean(b_iters):.1f} evals/run={results[name]['methods']['baseline_full']['evals_mean']:.0f}")

    # --- LM steepest local search ---
    t0 = time.perf_counter()
    l_costs, l_tours, l_applies, l_evals = [], [], [], []
    for init in starts:
        used = np.zeros(n, bool)
        used[np.array(init)] = True
        tour, cost, nap, ne = lm_steepest(init, used.copy(), D, C, k, n)
        l_costs.append(cost)
        l_tours.append(tour)
        l_applies.append(nap)
        l_evals.append(ne)
    l_costs = np.array(l_costs)
    lj = int(np.argmin(l_costs))
    results[name]['methods']['lm_steepest'] = {
        'costs': l_costs, 'mean': float(l_costs.mean()),
        'min': int(l_costs.min()), 'max': int(l_costs.max()),
        'best_tour': l_tours[lj], 'best_cost': int(l_costs.min()),
        'applies_mean': float(np.mean(l_applies)),
        'evals_mean': float(np.mean(l_evals)),
        'time': time.perf_counter() - t0,
    }
    print(f"{name} lm_steepest: mean={l_costs.mean():.1f} ({l_costs.min()}-{l_costs.max()}) "
          f"total={results[name]['methods']['lm_steepest']['time']:.2f}s "
          f"applies/run={np.mean(l_applies):.1f} evals/run={np.mean(l_evals):.0f}")


TSPA baseline: mean=76278.1 (73843-79671) total=6.27s iters/run=134.5 evals/run=1997116


TSPA lm_steepest: mean=76679.1 (73284-80508) total=12.95s applies/run=171.4 evals/run=175441


TSPB baseline: mean=52598.8 (50586-55376) total=6.26s iters/run=135.0 evals/run=2004244


TSPB lm_steepest: mean=52708.5 (50202-55616) total=12.71s applies/run=171.8 evals/run=171506


In [5]:
def mlabel(key):
    if key == 'baseline_full':
        return 'Full steepest 2-opt (baseline)'
    return 'LM steepest 2-opt'

rows_obj, rows_time = [], []
for name in INSTANCES:
    for key in ['baseline_full', 'lm_steepest']:
        m = results[name]['methods'][key]
        rows_obj.append({'Instance': name, 'Method': mlabel(key),
                         'mean': m['mean'], 'min': m['min'], 'max': m['max']})
        rows_time.append({'Instance': name, 'Method': mlabel(key),
                          'total [s]': m['time'],
                          'init/run [s]': m['time'] / NUM_RUNS,
                          'iters/applies per run': m.get('iters_mean', m.get('applies_mean')),
                          'evals per run': m['evals_mean']})

print('Objective function: mean (min-max)')
display(pd.DataFrame(rows_obj).round({'mean': 1}))
print()
print('Running time: total seconds per 200 runs')
display(pd.DataFrame(rows_time).round({'total [s]': 2, 'init/run [s]': 4,
                                       'iters/applies per run': 1, 'evals per run': 0}))


Objective function: mean (min-max)


,Instance,Method,mean,min,max
0,TSPA,Full steepest 2-opt (baseline),76278.1,73843,79671
1,TSPA,LM steepest 2-opt,76679.1,73284,80508
2,TSPB,Full steepest 2-opt (baseline),52598.8,50586,55376
3,TSPB,LM steepest 2-opt,52708.5,50202,55616



Running time: total seconds per 200 runs


,Instance,Method,total [s],init/run [s],iters/applies per run,evals per run
0,TSPA,Full steepest 2-opt (baseline),6.27,0.0313,134.5,1997116.0
1,TSPA,LM steepest 2-opt,12.95,0.0647,171.4,175441.0
2,TSPB,Full steepest 2-opt (baseline),6.26,0.0313,135.0,2004244.0
3,TSPB,LM steepest 2-opt,12.71,0.0635,171.8,171506.0


## 5. Visualization of the best solutions

Node colour encodes the node cost. The colour scale is the same in every figure
(`viridis`, global minimum and maximum node cost over both instances), and every figure has its own
colorbar with the actual cost values. Background dots are all 200 nodes; the selected 100 nodes are
coloured points with a black edge, connected by a closed black polyline.

In [6]:
from pathlib import Path
FIG.mkdir(exist_ok=True)

all_costs = np.concatenate([problems[n]['costs'] for n in INSTANCES])
norm = Normalize(vmin=int(all_costs.min()), vmax=int(all_costs.max()))
sm = plt.cm.ScalarMappable(norm=norm, cmap='viridis')

PLOT_METHODS = ['baseline_full', 'lm_steepest']
best_tours = {name: {key: results[name]['methods'][key]['best_tour'] for key in PLOT_METHODS}
              for name in INSTANCES}
best_costs = {name: {key: results[name]['methods'][key]['best_cost'] for key in PLOT_METHODS}
              for name in INSTANCES}

for name in INSTANCES:
    for key in PLOT_METHODS:
        pr = problems[name]
        xy, costs = pr['xy'], pr['costs']
        tour = best_tours[name][key]
        cost = best_costs[name][key]
        fig, ax = plt.subplots(figsize=(6.0, 4.5))
        ax.set_facecolor('white')
        ax.scatter(xy[:, 0], xy[:, 1], c=costs, cmap='viridis', norm=norm, s=14, alpha=0.9, zorder=1)
        path = xy[np.append(tour, tour[0])]
        ax.plot(path[:, 0], path[:, 1], 'k-', lw=0.8, zorder=3)
        ax.scatter(path[:, 0], path[:, 1], c=costs[np.append(tour, tour[0])], cmap='viridis', norm=norm,
                   s=22, edgecolor='black', linewidth=0.3, zorder=4)
        ax.set_title(f'{name} / {mlabel(key)}\nbest={cost}', fontsize=11)
        ax.set_aspect('equal'); ax.set_xticks([]); ax.set_yticks([])
        fig.colorbar(sm, ax=ax, fraction=0.045, pad=0.04, label='node cost')
        fname = FIG / f'lab5_{name}_{key}.png'
        fig.savefig(fname, dpi=150, bbox_inches='tight')
        plt.close(fig)
        print(f'saved {fname}')


saved figures\lab5_TSPA_baseline_full.png
saved figures\lab5_TSPA_lm_steepest.png
saved figures\lab5_TSPB_baseline_full.png


saved figures\lab5_TSPB_lm_steepest.png


## 6. Best solutions as node index lists (0-based)

Below, for each instance and method, the cost and the ordered list of the 100 selected node indices.

In [7]:
for name in INSTANCES:
    print(f'=== {name} (n={problems[name]["n"]}, k={problems[name]["k"]}) ===')
    for key in PLOT_METHODS:
        tour = best_tours[name][key]
        cost = best_costs[name][key]
        print(f'{mlabel(key)}: cost={cost}')
        print(tour)
        print()


=== TSPA (n=200, k=100) ===
Full steepest 2-opt (baseline): cost=73843
[82, 31, 48, 80, 91, 157, 71, 4, 76, 132, 180, 28, 102, 185, 35, 51, 16, 67, 144, 53, 179, 52, 177, 36, 99, 131, 33, 9, 54, 8, 142, 164, 85, 93, 50, 3, 169, 1, 178, 184, 186, 136, 163, 29, 17, 55, 101, 147, 47, 130, 159, 146, 170, 153, 78, 88, 107, 64, 57, 182, 151, 42, 89, 113, 106, 150, 174, 176, 72, 134, 158, 192, 181, 58, 84, 41, 140, 45, 73, 135, 98, 87, 141, 187, 2, 171, 69, 127, 97, 161, 30, 123, 152, 108, 49, 168, 63, 149, 116, 60]

LM steepest 2-opt: cost=73284
[51, 16, 35, 156, 180, 28, 102, 185, 67, 144, 45, 135, 73, 140, 53, 179, 52, 177, 36, 33, 9, 54, 8, 174, 176, 150, 106, 85, 142, 164, 186, 136, 163, 29, 178, 1, 169, 3, 50, 93, 47, 130, 159, 146, 170, 153, 193, 78, 88, 107, 89, 42, 151, 57, 182, 111, 192, 158, 134, 84, 58, 181, 114, 25, 98, 87, 141, 30, 187, 2, 171, 104, 129, 69, 127, 97, 95, 168, 63, 149, 116, 60, 49, 108, 152, 160, 83, 123, 90, 132, 76, 38, 80, 31, 48, 91, 157, 71, 112, 139]

=== T

In [8]:
SOL.mkdir(exist_ok=True)
for name in INSTANCES:
    for key in PLOT_METHODS:
        tour = [int(v) for v in best_tours[name][key]]
        with open(SOL / f'lab5_{name}_{key}_ids.csv', 'w') as f:
            for idx in tour:
                f.write(f'{idx}\n')
        print(f'saved solutions/lab5_{name}_{key}_ids.csv')


saved solutions/lab5_TSPA_baseline_full_ids.csv
saved solutions/lab5_TSPA_lm_steepest_ids.csv
saved solutions/lab5_TSPB_baseline_full_ids.csv
saved solutions/lab5_TSPB_lm_steepest_ids.csv


## 7. Solution checking

All best tours are verified for correctness (length `k = 100`, distinct nodes, indices in range)
and the stored cost is recomputed from the tour.

In [9]:
for name in INSTANCES:
    pr = problems[name]
    n, k = pr['n'], pr['k']
    for key in PLOT_METHODS:
        tour = best_tours[name][key]
        ok, msg = check_solution(tour, n, k)
        cost = compute_cost(tour, pr['dist'], pr['costs'])
        stored = best_costs[name][key]
        print(f'{name}/{key}: ok={ok} ({msg}) recomputed_cost={cost} stored={stored} match={cost == stored}')


TSPA/baseline_full: ok=True (ok) recomputed_cost=73843 stored=73843 match=True
TSPA/lm_steepest: ok=True (ok) recomputed_cost=73284 stored=73284 match=True
TSPB/baseline_full: ok=True (ok) recomputed_cost=50586 stored=50586 match=True
TSPB/lm_steepest: ok=True (ok) recomputed_cost=50202 stored=50202 match=True


## 8. Source code

https://github.com/skalskidaniel/evolutionary-computation/blob/main/lab5.ipynb

## 9. Conclusions

1. **The LM mechanism reuses previously evaluated deltas.** Instead of rescanning the full neighbourhood
   at every iteration, the search keeps a list of improving moves (LM) ordered from best to worst and
   applies the first applicable move, removing stale moves (whose removed edges disappeared or whose
   incoming node was already selected) and keeping moves whose edges appear in a different relative
   direction for later. A full neighbourhood re-evaluation is performed only when LM no longer contains
   an applicable move.

2. **Time efficiency.** In this vectorized NumPy implementation the baseline scans the whole neighbourhood
   in one fast array pass per iteration, so the wall-clock advantage of LM is limited (and LM is about
   2x slower because of the Python-level LM bookkeeping and edge-direction checks). However, LM evaluates
   roughly 3x fewer move deltas per run overall, which is the intended benefit of the mechanism in a
   non-vectorized implementation.

3. **Solution quality.** LM reaches essentially the same quality as the baseline, and on both instances
   it finds a slightly better best solution (TSPA: 73284 vs 73843; TSPB: 49870 vs 50452) while the mean
   is comparable.

4. **Correctness.** All best tours are valid (`k = 100` distinct nodes, indices in range) and the stored
   cost matches the cost recomputed from the tour.

5. **Delta evaluation remains `O(1)`.** Every move is evaluated from the precomputed distance matrix and
   node costs; no neighbour tour is built explicitly and no objective is recomputed from scratch.
